In [8]:
# 1 Question

import os
from dotenv import load_dotenv

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("Gemini API key not found in .env file")

from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(
    model="gemini-3.5-flash-lite",
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    model_info={
        "vision": True,
        "function_calling": True,
        "json_output": True,
        "family": "unknown",
        "structured_output": True
    }
)

from autogen_agentchat.agents import AssistantAgent

reviewer = AssistantAgent(
    name="Reviewer",
    model_client=model_client,
    system_message="You are a reviewer. Analyze the given content and provide constructive feedback."
)

editor = AssistantAgent(
    name="Editor",
    model_client=model_client,
    system_message="You are an editor. Improve the content based on the review feedback."
)

finalizer = AssistantAgent(
    name="Finalizer",
    model_client=model_client,
    system_message="You are a finalizer. Produce a polished final version. Return only the final content."
)

content = "Artificial Intelligence is changing the world. It helps people in many fields."

result1 = await reviewer.run(task=f"Review this content:\n{content}")
review = result1.messages[-1].content

result2 = await editor.run(
    task=f"Improve this content using the review:\n{review}\nOriginal content:\n{content}"
)
edited = result2.messages[-1].content

result3 = await finalizer.run(
    task=f"Finalize this edited content:\n{edited}"
)
final_output = result3.messages[-1].content

print("REVIEW:\n", review)
print("\nEDITED:\n", edited)
print("\nFINAL OUTPUT:\n", final_output)

REVIEW:
 Here is a constructive review of the content:

### **Overall Impression**
The content is grammatically correct and communicates a clear, accurate, and positive message about Artificial Intelligence (AI). However, it is **too brief, vague, and generic**. 

---

### **Strengths**
* **Clarity:** The sentences are simple and easy to understand.
* **Accuracy:** The core statement is undeniably true—AI is impacting the world across multiple domains.

---

### **Areas for Improvement**
* **Lack of Specificity:** Words like "changing the world," "helps people," and "many fields" are clichés. They don't tell the reader *how* AI is changing the world or *which* fields are benefiting.
* **Lack of Engagement:** Without concrete examples, statistics, or a unique perspective, the text fails to hook the reader's attention.
* **Tone:** It reads more like a basic placeholder (like "Lorem Ipsum") than a finished piece of writing.

---

### **Suggestions for Enhancement (Examples)**

Depending o

In [9]:
# 2 Question

translator = AssistantAgent(
    name="Translator",
    model_client=model_client,
    system_message=(
        "You are a professional translator. "
        "Translate the given English text into French accurately."
    )
)

culture_rephraser = AssistantAgent(
    name="CultureRephraser",
    model_client=model_client,
    system_message=(
        "You are a French language and cultural expert. "
        "Rewrite the French text naturally for French-speaking readers. "
        "Preserve its original meaning while improving cultural relevance."
    )
)

text = "Thank you for your support. We look forward to working with you."

result1 = await translator.run(task=text)
french_text = result1.messages[-1].content

result2 = await culture_rephraser.run(
    task=f"Make this French translation culturally natural:\n{french_text}"
)
final_text = result2.messages[-1].content

print("FRENCH TRANSLATION:")
print(french_text)

print("\nCULTURALLY REPHRASED:")
print(final_text)

FRENCH TRANSLATION:
Nous vous remercions pour votre soutien. Nous nous réjouissons à l'idée de collaborer avec vous.

CULTURALLY REPHRASED:
The original text is already grammatically correct, but it sounds a bit stiff and overly literal—especially "Nous nous réjouissons à l'idée de," which translates the English "We look forward to" a bit too mechanically for French business culture.

Here are a few ways to rewrite it so it sounds completely natural, depending on the context:

**1. Professional & Modern (Best for most business emails):**
> « Merci pour votre soutien. Nous avons hâte de collaborer avec vous. »
* *Why it works:* Replacing "Nous vous remercions" with "Merci" makes it warmer and less robotic, and "avoir hâte de" is the standard, natural French way to express anticipation.

**2. Polished & Corporate (For formal B2B relationships or official partners):**
> « Nous vous remercions chaleureusement pour votre soutien et avons hâte de travailler avec vous. »
* *Why it works:* Add

In [10]:
## 3 question 

generator = AssistantAgent(
    name="CodeGenerator",
    model_client=model_client,
    system_message=(
        "You are a Python developer. "
        "Generate clean and functional Python code for the given requirement. "
        "Return the complete code."
    )
)

reviewer = AssistantAgent(
    name="CodeReviewer",
    model_client=model_client,
    system_message=(
        "You are an experienced Python code reviewer. "
        "Check the code for syntax errors, logical errors, "
        "bugs, and possible improvements."
    )
)

bug_fixer = AssistantAgent(
    name="BugFixer",
    model_client=model_client,
    system_message=(
        "You are a Python debugging expert. "
        "Fix all issues identified by the reviewer. "
        "Return the complete corrected Python code."
    )
)

requirement = "Write a Python program to check whether a number is prime."

result1 = await generator.run(task=requirement)
generated_code = result1.messages[-1].content

result2 = await reviewer.run(
    task=f"Review this Python code and identify bugs:\n{generated_code}"
)
review = result2.messages[-1].content

result3 = await bug_fixer.run(
    task=(
        f"Original Python code:\n{generated_code}\n\n"
        f"Code review:\n{review}\n\n"
        "Fix the identified issues and return complete corrected code."
    )
)
fixed_code = result3.messages[-1].content

print("GENERATED CODE:")
print(generated_code)

print("\nCODE REVIEW:")
print(review)

print("\nFIXED CODE:")
print(fixed_code)

GENERATED CODE:
Here is a clean and efficient Python program to check whether a number is prime. It includes a function with optimized checks and a user input handler to test it.

```python
def is_prime(n):
    """Check if a number is a prime number."""
    # Numbers less than or equal to 1 are not prime
    if n <= 1:
        return False
    
    # 2 is the only even prime number
    if n == 2:
        return True
    
    # Exclude all other even numbers
    if n % 2 == 0:
        return False
    
    # Check odd factors up to the square root of n
    for i in range(3, int(n**0.5) + 1, 2):
        if n % i == 0:
            return False
            
    return True

# Example usage:
if __name__ == "__main__":
    try:
        user_input = int(input("Enter an integer to check if it's prime: "))
        if is_prime(user_input):
            print(f"{user_input} is a prime number.")
        else:
            print(f"{user_input} is not a prime number.")
    except ValueError:
        p

In [11]:
## 4 question
reader = AssistantAgent(
    name="ReaderAgent",
    model_client=model_client,
    system_message=(
        "You are an email analysis expert. "
        "Extract the sender's main point, request, "
        "important details, and required action from the email."
    )
)

responder = AssistantAgent(
    name="ResponderAgent",
    model_client=model_client,
    system_message=(
        "You are a professional email response assistant. "
        "Draft a polite and relevant reply based on the extracted information. "
        "Do not invent facts or commitments."
    )
)

formatter = AssistantAgent(
    name="FormatterAgent",
    model_client=model_client,
    system_message=(
        "You are an email formatting expert. "
        "Format the response with a suitable subject, greeting, "
        "professional body, and closing."
    )
)

email = """
Subject: Project Submission

Hello Team,

Please submit your project reports by Friday.
Also include the latest progress updates.

Regards,
Manager
"""

result1 = await reader.run(task=email)
main_point = result1.messages[-1].content

result2 = await responder.run(
    task=f"Draft a reply based on this email analysis:\n{main_point}"
)
reply = result2.messages[-1].content

result3 = await formatter.run(
    task=f"Format this reply as a professional email:\n{reply}"
)
final_email = result3.messages[-1].content

print("EMAIL ANALYSIS:")
print(main_point)

print("\nDRAFT REPLY:")
print(reply)

print("\nFORMATTED EMAIL:")
print(final_email)

EMAIL ANALYSIS:
Here is the analysis of the email:

*   **Sender's Main Point:** Project reports are due by the end of the week.
*   **Request:** Submit the project reports along with the latest progress updates.
*   **Important Details:** 
    *   Deadline: Friday.
    *   Required content: Project report and latest progress updates.
*   **Required Action:** Prepare and submit the project report and progress updates to the manager by Friday.

DRAFT REPLY:
Subject: Re: Project Reports Due

Hi [Manager's Name],

Thank you for the reminder. 

I am currently working on the project report and the latest progress updates, and I will ensure they are submitted to you by this Friday. 

Best regards,

[Your Name]

FORMATTED EMAIL:
Subject: Re: Project Reports Due

Dear [Manager's Name],

Thank you for the reminder. 

I am currently finalizing the project report along with the latest progress updates, and I will ensure they are submitted to you by this Friday. 

Please let me know if you need an

In [12]:

listener = AssistantAgent(
    name="ListenerAgent",
    model_client=model_client,
    system_message=(
        "You are a meeting analysis assistant. "
        "Extract the important discussion points, decisions, "
        "and responsibilities from the meeting transcript."
    )
)

action_agent = AssistantAgent(
    name="ActionAgent",
    model_client=model_client,
    system_message=(
        "You are a project coordinator. "
        "Convert the meeting discussion points into clear, "
        "specific, actionable to-do tasks."
    )
)

deadline_agent = AssistantAgent(
    name="DeadlineAgent",
    model_client=model_client,
    system_message=(
        "You are a project planning assistant. "
        "Suggest realistic timelines for each task. "
        "If no deadline is provided in the transcript, "
        "clearly label your timeline as a suggestion."
    )
)

transcript = """
The development team discussed the new website project.

Arun will complete the homepage design.
Priya will work on the database integration.
The testing team will test the application.
The team agreed to review the project progress next week.
"""

result1 = await listener.run(task=transcript)
discussion = result1.messages[-1].content

result2 = await action_agent.run(
    task=f"Convert these meeting points into actionable tasks:\n{discussion}"
)
actions = result2.messages[-1].content

result3 = await deadline_agent.run(
    task=(
        f"Assign suggested timelines to these tasks:\n{actions}\n\n"
        f"Original meeting transcript:\n{transcript}"
    )
)
final_tasks = result3.messages[-1].content

print("MEETING DISCUSSION:")
print(discussion)

print("\nACTION ITEMS:")
print(actions)

print("\nACTION ITEMS WITH TIMELINES:")
print(final_tasks)

MEETING DISCUSSION:
Here is the summary of the meeting based on the provided transcript:

### **Important Discussion Points**
* The team discussed the new website project.
* The team agreed to review the project progress next week.

### **Decisions Made**
* The team will hold a progress review meeting next week.

### **Responsibilities & Action Items**
* **Arun:** Complete the homepage design.
* **Priya:** Work on the database integration.
* **Testing Team:** Test the application.

ACTION ITEMS:
Here are the clear, specific, and actionable to-do tasks derived from the meeting summary:

* **Arun:** Complete the homepage design for the new website project.
* **Priya:** Work on and implement the database integration for the new website project.
* **Testing Team:** Perform testing on the application for the new website project.
* **Team Lead / Organizer:** Schedule and set up the project progress review meeting for next week.

ACTION ITEMS WITH TIMELINES:
Here are the suggested timelines f